# Train_Test_Split_CV Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Carve out a test set.** Four objects come back — feature/label pairs for train and test — and 80/20 leaves 455 rows to learn from, 114 to be judged on.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

**2. Same seed, same split.** Seeded shuffles are perfectly reproducible — teammates rerun your notebook and land on your exact numbers.

In [ ]:
import numpy as np

tr_a, _, y_a, _ = train_test_split(X, y, test_size=0.2, random_state=42)
tr_b, _, y_b, _ = train_test_split(X, y, test_size=0.2, random_state=42)
tr_c, _, _, _ = train_test_split(X, y, test_size=0.2, random_state=7)

print("seed 42 twice, identical rows?", np.array_equal(tr_a, tr_b))
print("seed 42 vs seed 7, differ?    ", not np.array_equal(tr_a, tr_c))

# Without a seed every rerun reshuffles: your accuracy changes, your bug
# "disappears", and nobody can reproduce anything. Seed the shuffle.

**3. Know your balance.** Roughly 63/37 benign/malignant — `stratify=y` is what pins those ratios into train and test alike.

In [ ]:
rates = pd.Series(y).value_counts(normalize=True)
for code, share in rates.items():
    print(f"class {code} ({cancer.target_names[code]:<11}) {share:.1%}")

# stratify=y copies the whole dataset's mix into every split, so the test
# set describes the same population you trained on.

## Part 2 — Practice

**4. Stratify under pressure.** The plain split drifts off 10%; the stratified split nails it — your test set stays a faithful miniature of the population.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
X_imb = rng.normal(size=(500, 2))
y_imb = np.concatenate([np.zeros(450, dtype=int), np.ones(50, dtype=int)])
rng.shuffle(y_imb)

_, te_plain, _, y_plain = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42)
_, te_strat, _, y_strat = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb)

print(f"FULL positive rate          : {y_imb.mean():.1%}")
print(f"TEST rate, plain split      : {y_plain.mean():.1%}")
print(f"TEST rate, stratified split : {y_strat.mean():.1%}")

**5. The iron rule, witnessed.** An unpruned tree aces the rows it studied and drops a few points on unseen ones — that drop is the generalisation story.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

train_acc = tree.score(X_train, y_train)
test_acc = tree.score(X_test, y_test)
print(f"train accuracy: {train_acc:.3f}   <- open-book exam")
print(f"test  accuracy: {test_acc:.3f}   <- closed-book exam")
print(f"gap           : {train_acc - test_acc:.3f}")

**6. Three rooms.** Tuning lives in validation, the final verdict comes from test — touched once, at the end.

In [ ]:
X_tv, X_test3, y_tv, y_test3 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train3, X_val, y_train3, y_val = train_test_split(
    X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)

for name, part in [("train", X_train3), ("validation", X_val),
                   ("test", X_test3)]:
    print(f"{name:<11}{len(part):>4} rows ({len(part) / len(X):.0%})")

**7. Five referees.** Five folds give five independent-ish verdicts; the mean is the skill, the std is the wobble.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

logreg = LogisticRegression(max_iter=5000)
scores = cross_val_score(logreg, X, y, cv=5)

print("fold accuracies:", scores.round(3))
print(f"summary: {scores.mean():.3f} +/- {scores.std():.3f}")

# Small std = the model performs consistently whichever fifth of the data
# it must predict blind. A big std warns the score was partly luck.

## Part 3 — Challenge

**8. A fair fight.** Same folds, same metric — only the model varies. KNN trails because raw units distort its distances; lesson 02's scaler is the fix.

In [ ]:
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

contenders = {
    "LogisticRegression": LogisticRegression(max_iter=5000),
    "KNN (k=5)":          KNeighborsClassifier(n_neighbors=5),
    "DecisionTree":       DecisionTreeClassifier(random_state=42),
}

rows = []
for name, model in contenders.items():
    s = cross_val_score(model, X, y, cv=5)
    rows.append({"model": name,
                 "mean_accuracy": round(s.mean(), 3),
                 "std": round(s.std(), 3)})

board = pd.DataFrame(rows).sort_values("mean_accuracy", ascending=False)
print(board.to_string(index=False))

# KNN measures distances, and unscaled columns shout at different volumes -
# scaling (lesson 02) closes the gap. Prefer the top mean but distrust a
# large std: swinging models may just be lucky here.

**9. Sorted data breaks naive folds.** Plain KFold slices sorted rows into all-zero and all-one islands; shuffling mixes them and stratifying pins every fold to the true 50/50.

In [ ]:
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold

y_sorted = np.concatenate([np.zeros(45, dtype=int), np.ones(45, dtype=int)])
X_dummy = np.zeros((len(y_sorted), 1))

splitters = {
    "plain KFold":   KFold(n_splits=5),
    "shuffled KFold": KFold(n_splits=5, shuffle=True, random_state=42),
    "StratifiedKFold": StratifiedKFold(n_splits=5, shuffle=True,
                                       random_state=42),
}
for name, splitter in splitters.items():
    rates = [y_sorted[test_idx].mean()
             for _, test_idx in splitter.split(X_dummy, y_sorted)]
    print(f"{name:<16}", [f"{r:.2f}" for r in rates])

# A fold with a 0.00 or 1.00 rate grades the model on a population that
# doesn't exist. Shuffle iid data, stratify classification - especially
# when positives are rare.